In [1]:
import glob
import pandas as pd

In [2]:
paths = glob.glob("data/mice_snapshot*/*.pkl")

frames = []
for path in paths:
    df = pd.read_pickle(path)
    df = df[df["subject_id"].astype(str) == "804924"]
    if not df.empty:
        df["source_file"] = path
        frames.append(df)

if not frames:
    print("No rows found for subject_id=804924.")
    raise SystemExit(0)

all_df = pd.concat(frames, ignore_index=True)

No rows found for subject_id=804924.


SystemExit: 0

/opt/conda/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3755: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [ ]:
# Detect date and stage columns if they exist
candidate_date_cols = [
    "session_date",
    "session_start_time",
    "session_start",
    "session_datetime",
    "session_time",
    "date",
]
candidate_stage_cols = [
    "stage",
    "curriculum_name",
    "training_stage",
    "curriculum",
]

date_col = next((c for c in candidate_date_cols if c in all_df.columns), None)
stage_col = next((c for c in candidate_stage_cols if c in all_df.columns), None)

keep_cols = ["ses_idx"]
if "session_name" in all_df.columns:
    keep_cols.append("session_name")
if date_col:
    all_df[date_col] = pd.to_datetime(all_df[date_col], errors="coerce")
    keep_cols.append(date_col)
if stage_col:
    keep_cols.append(stage_col)

if date_col:
    all_df = all_df.sort_values([date_col, "ses_idx"]).reset_index(drop=True)
else:
    all_df = all_df.sort_values(["ses_idx"]).reset_index(drop=True)

print(all_df.to_string(index=False))